# Kiểm thử file → tiền xử lý độc lập

Notebook dùng DOCX và ảnh giả lập. Không gọi AWS/model. Trích xuất DOCX và đọc ảnh dùng thư viện thật; cell OCR có engine giả được ghi rõ. Chọn kernel môi trường project đã `pip install -e .`.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "input_processing/pipeline.py").is_file())
if str(root) not in sys.path: sys.path.insert(0, str(root))
from tempfile import TemporaryDirectory
from docx import Document
from input_processing import process_job, RetrievedFile


In [ ]:
with TemporaryDirectory() as directory:
    folder = Path(directory)
    document = Document()
    document.add_paragraph("Điều 1. Thanh toán")
    document.add_paragraph("Bên A thanh toán trong 30 ngày.")
    table = document.add_table(rows=1, cols=3)
    table.cell(0, 0).text = "Dịch vụ"
    table.cell(0, 2).text = "1000"
    document.add_paragraph("Điều 2. Thời hạn")
    document.add_paragraph("Hợp đồng có thời hạn một năm.")
    document.save(folder / "demo.docx")
    file = RetrievedFile((folder / "demo.docx").read_bytes(), "demo.docx", {"type": "fixture"})
    result = process_job({"job_id": "notebook-file", "contract_id": "DEMO"},
        file=file)
assert result["status"] == "ok", result
assert result["document"]["format"] == "docx"
assert result["document"]["page_count"] is None
result["stats"], result["warnings"]

In [ ]:
assert result["stats"]["chunk_count"] == len(result["chunks"])
assert all(c["source_spans"] for c in result["chunks"])
assert all(c["document"]["format"] == "docx" for c in result["chunks"])
print(result["chunks"][0]["text"])
result["document"]


## OCR giả lập để kiểm tra nối nguồn

Cell này kiểm tra giao diện engine và metadata ảnh, không kiểm tra độ chính xác nhận dạng. Để dùng thật, cài Tesseract cùng language packs `vie`, `eng`, rồi bỏ `ocr_engine=DemoOCR()`. Notebook kết thúc ở output tiền xử lý; không nối agent.

In [ ]:
from PIL import Image
class DemoOCR:
    def recognize(self, image, *, languages, timeout):
        return "Điều 1. Nội dung giả lập OCR\nThanh toán trong 30 ngày."
with TemporaryDirectory() as directory:
    folder = Path(directory)
    with Image.new("RGB", (320, 120), "white") as image:
        image.save(folder / "scan.png")
    file = RetrievedFile((folder / "scan.png").read_bytes(), "scan.png", {"type": "fixture"})
    scanned = process_job({"job_id": "demo-scan", "contract_id": "SCAN"},
        file=file, ocr_engine=DemoOCR())
assert scanned["status"] == "ok", scanned
assert scanned["extraction"]["ocr_segments"] == 1
assert scanned["chunks"][0]["pages"] == [1]
print("DOCX and standalone source manifest checks passed; OCR engine was simulated.")